# Final project model

Selected course model: application-only, gender excluded, age retained. Dilraj's richer application features and regularized neural network are evaluated using Gil's fixed splits, threshold selection and fairness workflow. Stanley's work provides the simple neural-network reference and related regularization experiments. This is one network, not an ensemble of teammates' models.

Run from the repository root with application_train.csv available. The shipped checkpoint is seed42, epoch23; blackbox F1 selected cutoff0.167. The default run evaluates that frozen checkpoint. Training code is in final_project/model.py; setting RETRAIN=True creates separate outputs and does not overwrite the selected model or its error CSV.

Selection was frozen before this model's eyeball/test evaluation. Earlier project exposure remains; these are exploratory course results, not a pristine prospective test or deployment certification. Team review of draft error notes remains pending.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from final_project.model import run_main, run_alternative_splits
RETRAIN = False


## 1 Selected recipe and selection evidence

In [2]:
display(pd.DataFrame(json.loads(Path('peer_review/20261014/summary.json').read_text())))
print(Path('final_project/selection_decision.json').read_text())

{
  "date": "2026-10-14",
  "selected": "peer_recipe",
  "seed": 42,
  "threshold": 0.167,
  "epoch": 23,
  "selection_metric": "blackbox AUC",
  "threshold_metric": "blackbox F1",
  "mean_auc_gain": 0.0078864204937935,
  "seed_wins": 3,
  "decision": "Adopt for course project within application-only gender-excluded scope. Better ranking/F1; accept documented recall and disparity tradeoffs without claiming fairness approval.",
  "fairness": "No quantitative acceptance tolerance agreed. Report both own-F1 and matched-FPR disparities; not a deployment certification.",
  "test_or_eyeball_used_for_this_selection": false,
  "historical_exposure": true
}


,name,auc,f1,recall,precision,fpr,gender_gap_pp,age_gap_pp,auc_gain,seed_wins
0,baseline,0.747075,0.296473,0.440745,0.223379,0.134583,5.670490,22.307376,0.000000,0
1,quantile_current,0.745818,0.291678,0.411574,0.226161,0.123901,5.708702,21.926937,-0.001257,0
2,peer_recipe,0.754961,0.302354,0.411641,0.239372,0.115193,5.909904,20.639106,0.007886,3


## 2 Evaluate the selected checkpoint

Features are imputed and quantile-transformed using training rows only. Missingness, categorical encoding and constant/duplicate removal are also learned from training. Engineered inputs represent affordability, external-score combinations and other application properties. The128/64 network includes a linear skip connection, batch normalization, SiLU and dropout. AdamW, input noise, gradient clipping and a learning-rate scheduler control fitting. Blackbox AUC selects the checkpoint; blackbox F1 selects the operating cutoff.


In [3]:
metrics = run_main(retrain=RETRAIN)
display(metrics.round(5))

,partition,threshold,selected_epoch,n,prevalence,auc,average_precision,accuracy,precision,recall,f1,false_positive_rate,brier,log_loss,tn,fp,fn,tp
0,train,0.167,23,184506,0.08073,0.77635,0.26451,0.85919,0.26434,0.41739,0.32368,0.10201,0.06673,0.24065,152309,17302,8678,6217
1,blackbox,0.167,23,61002,0.08074,0.75473,0.23643,0.85255,0.24473,0.39614,0.30255,0.10737,0.06819,0.24740,50056,6021,2974,1951
2,eyeball,0.167,23,500,0.08000,0.68652,0.15903,0.83800,0.18462,0.30000,0.22857,0.11522,0.07254,0.26576,407,53,28,12
3,test,0.167,23,61503,0.08073,0.75668,0.24384,0.85511,0.24975,0.39658,0.30648,0.10462,0.06785,0.24639,50623,5915,2996,1969


## 3 Fairness at the frozen threshold

In [4]:
fairness=pd.read_json('final_project/selected/blackbox_fairness.json')
display(fairness[['attribute','group','n','false_positive_rate','recall','precision','npv','favorable_prediction_rate','small_group_warning']].round(4))
display(pd.read_json('final_project/selected/blackbox_fairness_gaps.json').round(3))

,attribute,group,n,false_positive_rate,recall,precision,npv,favorable_prediction_rate,small_group_warning
0,CODE_GENDER,F,40195,0.0890,0.3567,0.2323,0.9494,0.8923,False
1,CODE_GENDER,M,20806,0.1441,0.4489,0.2596,0.9324,0.8251,False
2,CODE_GENDER,XNA,1,1.0000,NaN,0.0000,NaN,0.0000,True
3,age_group,Under 30,8900,0.2202,0.5511,0.2420,0.9316,0.7424,False
4,age_group,30 to under 45,24675,0.1235,0.4230,0.2554,0.9382,0.8492,False
5,age_group,45 to under 60,20531,0.0706,0.3037,0.2311,0.9503,0.9142,False
6,age_group,60 and older,6896,0.0242,0.1175,0.1970,0.9563,0.9713,False


,attribute,metric,eligible_groups,excluded_groups,lowest_group,highest_group,minimum,maximum,gap_pp
0,CODE_GENDER,false_positive_rate,2,XNA,F,M,0.089,0.144,5.515
1,CODE_GENDER,recall,2,XNA,F,M,0.357,0.449,9.219
2,CODE_GENDER,precision,2,XNA,F,M,0.232,0.260,2.734
3,CODE_GENDER,npv,2,XNA,M,F,0.932,0.949,1.700
4,CODE_GENDER,favorable_prediction_rate,2,XNA,M,F,0.825,0.892,6.720
5,CODE_GENDER,specificity,2,XNA,M,F,0.856,0.911,5.515
6,age_group,false_positive_rate,4,,60 and older,Under 30,0.024,0.220,19.597
7,age_group,recall,4,,60 and older,Under 30,0.117,0.551,43.367
8,age_group,precision,4,,60 and older,30 to under 45,0.197,0.255,5.841
9,age_group,npv,4,,Under 30,60 and older,0.932,0.956,2.467


Gender removal does not guarantee fairness. Favorable prediction means predicted no difficulty, not loan approval. FPR disparities concern false alarms; recall concerns detected difficulties. Report counts and uncertainty. No approved disparity tolerance or business cost matrix has been supplied. At matched overall FPR, the candidate has slightly higher mean recall but wider mean gender and age FPR gaps than the previous model.

## 4 Three evaluation protocols

Stratified remains the main source-population evaluation. Random sampling preserves the same partition counts without target stratification. The nonrandom scenario trains/dev-tunes on people with earlier Home Credit application records and tests on those without. previous_application.csv defines membership only; it adds no predictors. All500 eyeball applicants are excluded from both alternate protocols. Different populations and prevalence mean these scores do not identify the best split.

Saved comparisons are displayed by default. Set RERUN_SPLITS=True to retrain the two alternate protocols using previous_application.csv. Each chooses its own checkpoint and cutoff on its own dev set before test scoring.


In [5]:
RERUN_SPLITS=False
if RERUN_SPLITS:
    comparison=run_alternative_splits()
else:
    comparison=pd.read_json('final_project/split_comparison.json')
display(comparison[['protocol','partition','n','prevalence','auc','accuracy','precision','recall','f1','false_positive_rate','threshold']].round(4))

,protocol,partition,n,prevalence,auc,accuracy,precision,recall,f1,false_positive_rate,threshold
0,stratified,train,184506,0.0807,0.7763,0.8592,0.2643,0.4174,0.3237,0.1020,0.167
1,stratified,blackbox,61002,0.0807,0.7547,0.8525,0.2447,0.3961,0.3026,0.1074,0.167
2,stratified,eyeball,500,0.0800,0.6865,0.8380,0.1846,0.3000,0.2286,0.1152,0.167
3,stratified,test,61503,0.0807,0.7567,0.8551,0.2497,0.3966,0.3065,0.1046,0.167
4,random,train,184506,0.0810,0.7814,0.8567,0.2664,0.4387,0.3315,0.1065,0.166
5,random,blackbox,61002,0.0795,0.7551,0.8526,0.2435,0.4056,0.3043,0.1088,0.166
6,random,test,61503,0.0811,0.7572,0.8504,0.2443,0.4032,0.3043,0.1101,0.166
7,no_previous_application,train,232472,0.0819,0.7744,0.8334,0.2417,0.4835,0.3223,0.1354,0.134
8,no_previous_application,blackbox,58118,0.0819,0.7558,0.8314,0.2325,0.4598,0.3088,0.1355,0.134
9,no_previous_application,test,16421,0.0596,0.7637,0.8953,0.2245,0.3088,0.2600,0.0675,0.134


## 5 Next file

Open Final_Error_Analysis.ipynb to export all errors from this checkpoint on the fixed eyeball pool, review correct-case comparisons, and repeat the nine recorded hypothesis checks on blackbox aggregates. The final CSV is Final_Error_Analysis.csv. It must remain attached to this model and cutoff; a retrained or replaced model needs a new review version.
